In [1]:
import os

os.environ['http_proxy'] = "http://xen03.iitd.ac.in:3128"
os.environ['https_proxy'] = "http://xen03.iitd.ac.in:3128"

In [2]:
import datasets
from datasets import Dataset
data = datasets.load_dataset("pminervini/HaluEval", "dialogue", split='data')

/home/anwoy/phuhoang/miniconda3/envs/HIDE/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Generating data split: 100%|██████████| 10000/10000 [00:00<00:00, 311157.07 examples/s]


In [3]:
data

Dataset({
    features: ['knowledge', 'dialogue_history', 'right_response', 'hallucinated_response'],
    num_rows: 10000
})

In [5]:
data[0]["dialogue_history"]

'[Human]: Do you like Iron Man [Assistant]: Sure do! Robert Downey Jr. is a favorite. [Human]: Yes i like him too did you know he also was in Zodiac a crime fiction film. '

In [6]:
id_map = {_['dialogue_history']:str(i) for i, _ in enumerate(data)}

In [7]:
len(id_map)

10000

PROBE

In [11]:
import os
import shutil
import tempfile
from pathlib import Path
from typing import Optional, Tuple, Union

import torch
from huggingface_hub import HfApi, hf_hub_download, login
from huggingface_hub.utils import validate_repo_id

LOCAL_PROBES_DIR = Path("/home/anwoy/HIDE/data/probes")
def download_probe_from_hf(
    repo_id: str,
    probe_id: Optional[str] = None,
    local_folder: Optional[Union[str, Path]] = None,
    hf_repo_subfolder_prefix: str = "",
    token: Optional[str] = None
) -> None:
    """Simplified probe download function for Modal."""
    api = HfApi()

    if local_folder is None:
        local_folder = LOCAL_PROBES_DIR / probe_id
    elif isinstance(local_folder, str):
        local_folder = Path(local_folder)

    local_folder.mkdir(parents=True, exist_ok=True)
    
    # List files in the repository subfolder
    repo_files = api.list_repo_files(
        repo_id=repo_id,
        repo_type="model",
        revision="main"
    )
    print(f"Files in repo: {repo_files}")
    
    # Filter files by subfolder
    path_in_repo = os.path.join(hf_repo_subfolder_prefix, probe_id)
    subfolder_files = [f for f in repo_files if f.startswith(f"{path_in_repo}/")]
    print(f"Files in subfolder: {subfolder_files}")
    # Download each file
    for file_path in subfolder_files:
        print(file_path)
        # Get relative path within subfolder
        relative_path = file_path[len(path_in_repo):].lstrip('/')
        
        # Create subdirectory if needed
        local_file_path = local_folder / relative_path
        local_file_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Download file
        downloaded_file = hf_hub_download(
            repo_id=repo_id,
            filename=file_path,
            token=token
        )
        
        # Copy to destination
        shutil.copy(downloaded_file, local_file_path)
    
    print(f"Downloaded probe to {local_folder}")



In [15]:
REPO_ID = "obalcells/hallucination-probes"
PROBE_ID = "llama3_1_8b_linear"
# Default directory for saving probes locally
LOCAL_PROBES_DIR = Path("/home/anwoy/HIDE/data/probes")

download_probe_from_hf(
            repo_id=REPO_ID,
            probe_id=PROBE_ID
        )

Files in repo: ['.gitattributes', 'README.md', 'gemma2_9b_linear/eval_metrics.jsonl', 'gemma2_9b_linear/eval_metrics_gemma2_9b_it_linear_n=2k.json', 'gemma2_9b_linear/gemma2_9b_longfact_augmented_test_roc_curves.png', 'gemma2_9b_linear/gemma2_9b_longfact_test_roc_curves.png', 'gemma2_9b_linear/llama3_1_8b_validation_roc_curves.png', 'gemma2_9b_linear/probe_config.json', 'gemma2_9b_linear/probe_head.bin', 'gemma2_9b_linear/special_tokens_map.json', 'gemma2_9b_linear/tokenizer.json', 'gemma2_9b_linear/tokenizer.model', 'gemma2_9b_linear/tokenizer_config.json', 'gemma2_9b_linear/training_config.json', 'gemma2_9b_lora_lambda_kl_0_05/README.md', 'gemma2_9b_lora_lambda_kl_0_05/adapter_config.json', 'gemma2_9b_lora_lambda_kl_0_05/adapter_model.safetensors', 'gemma2_9b_lora_lambda_kl_0_05/eval_metrics.jsonl', 'gemma2_9b_lora_lambda_kl_0_05/eval_metrics_gemma2_9b_lora_lambda_kl=0.5.json', 'gemma2_9b_lora_lambda_kl_0_05/gemma2_9b_longfact_augmented_test_roc_curves.png', 'gemma2_9b_lora_lambda_kl

In [1]:
import torch
import torch.nn as nn
import json
from pathlib import Path
from typing import Optional, Tuple, Union


def load_probe_head(
    probe_dir: Path,
    dtype: torch.dtype = torch.bfloat16,
    device: str = 'cuda'
) -> Tuple[nn.Module, int]:
    """Load probe head from disk."""
    # Load probe config
    with open(probe_dir / "probe_config.json") as f:
        probe_config = json.load(f)
    
    hidden_size = probe_config['hidden_size']
    probe_layer_idx = probe_config['layer_idx']
    
    # Create probe head
    probe_head = nn.Linear(hidden_size, 1, device=device, dtype=dtype)
    
    # Load weights
    state_dict = torch.load(
        probe_dir / "probe_head.bin",
        map_location="cpu",
        weights_only=True
    )
    probe_head.load_state_dict(state_dict)
    probe_head.eval()
    
    return probe_head, probe_layer_idx

probe_dir = Path("/home/anwoy/HIDE/data/probes/llama3_1_8b_linear")
probe_head, probe_layer_idx = load_probe_head(probe_dir)

In [2]:
probe_layer_idx

30

In [3]:
import models
model, tokenizer = models.load_model_and_tokenizer("llama3-8b", "cuda:0")

/home/anwoy/phuhoang/miniconda3/envs/HIDE/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 4/4 [00:58<00:00, 14.51s/it]


In [4]:
device = "cuda:0"
probe_head = probe_head.to(device)
text = "What is the capital of France"
inputs = tokenizer(text, return_tensors="pt").to(device)

# 4. RUN MODEL & EXTRACT HIDDEN STATES
# We don't need the final text, we need the internal brain activity.
with torch.no_grad():
    outputs = model(
        **inputs, 
        output_hidden_states=True # <--- CRITICAL: This asks HF to return internal layers
    )

target_layer_activations = outputs.hidden_states[probe_layer_idx]

input_dtype = target_layer_activations.dtype

# 2. Force the probe to match that data type
probe_head = probe_head.to(dtype=input_dtype)

# 6. APPLY THE PROBE
# activations shape: [batch_size, seq_len, hidden_size]
probe_score = probe_head(target_layer_activations) 

# Convert raw score (logit) to probability (0 to 1)
probe_probability = torch.sigmoid(probe_score)



In [7]:
len(outputs.hidden_states)

33

In [5]:
selected_states = [token_tuple[probe_layer_idx] for token_tuple in outputs.hidden_states]
    
X = selected_states[0][0,:,:]
X = X.to(torch.float32)

IndexError: index 30 is out of bounds for dimension 0 with size 1

In [17]:
# torch.max(probe_probability)
# probe_score
# target_layer_activations.shape
probe_probability

tensor([[[1.0000],
         [0.5752],
         [0.4661],
         [0.3936],
         [0.5000],
         [0.6978],
         [0.2651]]], device='cuda:0', dtype=torch.float16,
       grad_fn=<SigmoidBackward0>)

In [9]:
# 7. INTERPRET RESULTS
# The probe gives a score for EVERY token in the sequence.
tokens = tokenizer.convert_ids_to_tokens(inputs.input_ids[0])
probs = probe_probability.squeeze().tolist()

print("\nProbe Activation per token:")
for token, prob in zip(tokens, probs):
    # Usually: High prob (near 1) = True/Factual, Low prob (near 0) = False/Hallucination
    # (You must verify the training labels of your specific probe to be sure)
    print(f"{token:>15} : {prob:.4f}")


Probe Activation per token:
<|begin_of_text|> : 1.0000
           What : 0.5752
            Ġis : 0.4661
           Ġthe : 0.3936
       Ġcapital : 0.5000
            Ġof : 0.6978
        ĠFrance : 0.2651


In [ ]:
import pickle as pkl
file_name = "/home/anwoy/HIDE/data/output/ablations/llama3-8b_SQuAD_1/0.pkl"
print(file_name)
f = open(file_name, "rb")
resultDict = pkl.load(f)

/home/anwoy/HIDE/data/output/ablations/llama3-8b_SQuAD_1/0.pkl


In [10]:
# input_length = []
output_length_list = []
output_tokens_topk_list = []
single_gen_time_list = []
hide_time_list = []
for item in resultDict:
    try:
        output_length = len(item['most_likely_generation_ids'])
        op_tokens_len = len(item['output_tokens_topk'])
        single_gen_time = item['greedy_generation_time']
        hide_time = item['hsic_time']
        output_length_list.append(output_length)
        output_tokens_topk_list.append(op_tokens_len)
        single_gen_time_list.append(single_gen_time)
        hide_time_list.append(hide_time)
    except:
        continue

# resultDict[0]

In [11]:
# resultDict[0]

In [12]:
# len(output_length)
import pandas as pd
print(pd.Series(output_tokens_topk_list).describe(percentiles=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]))

# Output will include:
# count, mean, std, min, 25%, 50% (median), 75%, max

count    5918.000000
mean       11.612200
std         7.129207
min         1.000000
10%         2.000000
20%         4.000000
30%         5.000000
40%         8.000000
50%        11.000000
60%        15.000000
70%        20.000000
80%        20.000000
90%        20.000000
max        20.000000
dtype: float64


In [9]:
count = 0
single_gen=0
hsic_time = 0
for item in resultDict:
    single_gen+=item['greedy_generation_time']
    hsic_time+=item['hsic_time']
    count+=1    

print(f"Average greedy generation time: {single_gen/count:.4f} seconds")
print(f"Average HSIC time: {hsic_time/count:.4f} seconds")

Average greedy generation time: 1.0622 seconds
Average HSIC time: 0.2125 seconds


In [1]:
# SPDX-License-Identifier: Apache-2.0
# SPDX-FileCopyrightText: Copyright contributors to the vLLM project
import tempfile

from safetensors import safe_open

from vllm import LLM, SamplingParams

# Example: Using the custom "extract_hidden_states" speculator method and
# ExampleHiddenStatesConnector to extract and save hidden states from vllm

with tempfile.TemporaryDirectory() as tmpdirname:
    llm = LLM(
        model="/home/models/Meta-Llama-3-8B",  # Your target model
        speculative_config={
            "method": "extract_hidden_states",
            "num_speculative_tokens": 1,
            "draft_model_config": {
                "hf_config": {
                    "eagle_aux_hidden_state_layer_ids": [  # Target model layer indices
                        1,
                        2,
                        3,
                        4,
                    ],
                }
            },
        },
        kv_transfer_config={
            "kv_connector": "ExampleHiddenStatesConnector",
            "kv_role": "kv_producer",
            "kv_connector_extra_config": {
                "shared_storage_path": tmpdirname,
            },
        },
    )

    prompts = ["Generate a sentence with hidden states", "Write a python function"]
    sampling_params = SamplingParams(max_tokens=1)
    outputs = llm.generate(prompts, sampling_params)

    for output in outputs:
        print("\nPrompt:", output.prompt)
        print("Prompt token ids:", output.prompt_token_ids)

        hidden_states_path = output.kv_transfer_params.get("hidden_states_path")
        assert hidden_states_path is not None
        print("Prompt hidden states path:", hidden_states_path)

        with safe_open(hidden_states_path, "pt") as f:
            # print(f)
            print(list(f.keys()))
            token_ids = f.get_tensor("token_ids")
            hidden_states = f.get_tensor("hidden_states")

            print("Extracted token ids:", token_ids)  # Matches prompt token ids
            print(
                "Extracted hidden states shape:", hidden_states.shape
            )  # [num_hidden_layers, prompt len, hidden size]
            print("Extracted hidden states:", hidden_states)

/home/anwoy/phuhoang/miniconda3/envs/HIDE/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 03-28 19:13:36 [utils.py:233] non-default args: {'disable_log_stats': True, 'speculative_config': {'method': 'extract_hidden_states', 'num_speculative_tokens': 1, 'draft_model_config': {'hf_config': {'eagle_aux_hidden_state_layer_ids': [1, 2, 3, 4]}}}, 'kv_transfer_config': KVTransferConfig(kv_connector='ExampleHiddenStatesConnector', engine_id='2000c589-70d9-432b-bf60-e5635f02a828', kv_buffer_device='cuda', kv_buffer_size=1000000000.0, kv_role='kv_producer', kv_rank=None, kv_parallel_size=1, kv_ip='127.0.0.1', kv_port=14579, kv_connector_extra_config={'shared_storage_path': '/tmp/tmpsivuvymg'}, kv_connector_module_path=None, enable_permute_local_kv=False, kv_load_failure_policy='fail'), 'model': '/home/models/Meta-Llama-3-8B'}
INFO 03-28 19:13:36 [model.py:533] Resolved architecture: LlamaForCausalLM
INFO 03-28 19:13:36 [model.py:1582] Using max model len 8192
INFO 03-28 19:13:36 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 03-28 19:13:36 [

(EngineCore pid=1366698) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(EngineCore pid=1366698) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:01<00:04,  1.57s/it]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:03<00:03,  1.67s/it]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:04<00:01,  1.67s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:05<00:00,  1.19s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:05<00:00,  1.36s/it]
(EngineCore pid=1366698) 


(EngineCore pid=1366698) INFO 03-28 19:13:46 [default_loader.py:384] Loading weights took 5.47 seconds
(EngineCore pid=1366698) INFO 03-28 19:13:46 [gpu_model_runner.py:4505] Loading drafter model...
(EngineCore pid=1366698) WARNING 03-28 19:13:46 [vllm.py:1739] `torch.compile` is turned on, but the model /home/models/Meta-Llama-3-8B does not support it. Please open an issue on GitHub if you want it to be supported.
(EngineCore pid=1366698) INFO 03-28 19:13:46 [default_loader.py:384] Loading weights took 1060064.72 seconds
(EngineCore pid=1366698) INFO 03-28 19:13:46 [gpu_model_runner.py:4543] Using auxiliary layers from speculative config: (1, 2, 3, 4)
(EngineCore pid=1366698) INFO 03-28 19:13:47 [gpu_model_runner.py:4566] Model loading took 14.96 GiB memory and 6.244596 seconds
(EngineCore pid=1366698) INFO 03-28 19:13:51 [backends.py:988] Using cache directory: /home/anwoy/.cache/vllm/torch_compile_cache/2949ac8aa2/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=1366698) 

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:02<00:00, 18.91it/s]


(EngineCore pid=1366698) INFO 03-28 19:14:01 [gpu_model_runner.py:5746] Graph capturing finished in 4 secs, took 0.39 GiB
(EngineCore pid=1366698) INFO 03-28 19:14:01 [gpu_worker.py:617] CUDA graph pool memory: 0.39 GiB (actual), 0.41 GiB (estimated), difference: 0.02 GiB (4.0%).
(EngineCore pid=1366698) INFO 03-28 19:14:01 [core.py:281] init engine (profile, create kv cache, warmup model) took 14.03 seconds
(EngineCore pid=1366698) INFO 03-28 19:14:02 [factory.py:64] Creating v1 connector with name: ExampleHiddenStatesConnector and engine_id: 2000c589-70d9-432b-bf60-e5635f02a828
(EngineCore pid=1366698) WARNING 03-28 19:14:02 [base.py:189] Initializing KVConnectorBase_V1. This API is experimental and subject to change in the future as we iterate the design.
(EngineCore pid=1366698) INFO 03-28 19:14:02 [example_hidden_states_connector.py:135] KVTransferConfig(kv_connector='ExampleHiddenStatesConnector', engine_id='2000c589-70d9-432b-bf60-e5635f02a828', kv_buffer_device='cuda', kv_buffe

Processed prompts: 100%|██████████| 2/2 [00:00<00:00, 31.38it/s, est. speed input: 192.20 toks/s, output: 32.03 toks/s]


Prompt: Generate a sentence with hidden states
Prompt token ids: [128000, 32215, 264, 11914, 449, 8340, 5415]
Prompt hidden states path: /tmp/tmpsivuvymg/0-872b0067.safetensors
['hidden_states', 'token_ids']
Extracted token ids: tensor([128000,  32215,    264,  11914,    449,   8340,   5415])
Extracted hidden states shape: torch.Size([7, 4, 4096])
Extracted hidden states: tensor([[[ 2.0142e-03,  2.1057e-03, -4.8828e-03,  ...,  1.3000e-02,
           3.1891e-03, -5.8289e-03],
         [-1.3123e-02,  4.2725e-02, -1.5869e-02,  ...,  4.5508e-01,
           1.8750e-01, -4.7119e-02],
         [-1.6235e-02,  4.6875e-02, -1.1719e-02,  ...,  4.6484e-01,
           1.9727e-01, -5.3223e-02],
         [-1.2207e-03,  5.3223e-02,  6.3477e-03,  ...,  5.0391e-01,
           1.9336e-01, -5.6641e-02]],

        [[ 1.6846e-02,  4.7607e-03, -3.1494e-02,  ..., -4.5166e-02,
          -1.1475e-02,  4.5776e-03],
         [ 1.6846e-02,  4.1809e-03, -3.7842e-02,  ..., -5.7861e-02,
          -2.6611e-02, -1.208

In [3]:
hidden_states_path = output.kv_transfer_params.get("hidden_states_path")
assert hidden_states_path is not None
print("Prompt hidden states path:", hidden_states_path)
with safe_open(hidden_states_path, "pt") as f:
            print(f)
            # token_ids = f.get_tensor("token_ids")
            # hidden_states = f.get_tensor("hidden_states")

            # print("Extracted token ids:", token_ids)  # Matches prompt token ids
            # print(
            #     "Extracted hidden states shape:", hidden_states.shape
            # )  # [num_hidden_layers, prompt len, hidden size]
            # print("Extracted hidden states:", hidden_states)

Prompt hidden states path: /tmp/tmp0p3liug1/1-837bc594.safetensors


FileNotFoundError: No such file or directory: "/tmp/tmp0p3liug1/1-837bc594.safetensors"

In [ ]:
outputs[0].outputs[0]
# text, token_ids

CompletionOutput(index=0, text='. The sentence predicator has no data. It is mode executes only for generating the suggestion data. When the sentence predicator starts, it initializes with random data.\n\n# Load the required data from the pretrained models \n\nimport sys\nimport numpy as np\nimport pandas as pd\nfrom utils import *\nimport time\nfrom vectorize.deck_util import col_mapping\nfrom intent.v2.intent.word_vectorize.precomputation.concept2concept import VectorizeConcept2Concept\nfrom intent.prediction.pretrainedIntent.predicator.preIndicationPredicator import MetaPreengPredictionPredicator\nfrom intent.prediction.pretrainedIntent.predicator.predicator import', token_ids=[13, 578, 11914, 4255, 13557, 706, 912, 828, 13, 1102, 374, 3941, 52535, 1193, 369, 24038, 279, 24710, 828, 13, 3277, 279, 11914, 4255, 13557, 8638, 11, 433, 58957, 449, 4288, 828, 382, 2, 9069, 279, 2631, 828, 505, 279, 81769, 4211, 4815, 475, 5826, 198, 475, 8760, 439, 2660, 198, 475, 19130, 439, 7900, 198, 

In [16]:
hidden_states_path = outputs[0].kv_transfer_params.get("hidden_states_path")

In [15]:
print("Prompt hidden states path:", hidden_states_path)

with safe_open(hidden_states_path, "pt") as f:
    print(f)
    # token_ids = f.get_tensor("token_ids")
    # hidden_states = f.get_tensor("hidden_states")

    # print("Extracted token ids:", token_ids)  # Matches prompt token ids
    # print(
    #     "Extracted hidden states shape:", hidden_states.shape
    # )  # [num_hidden_layers, prompt len, hidden size]
    # print("Extracted hidden states:", hidden_states)

Prompt hidden states path: /tmp/tmpe3vf20_y/0-a6f5f2b1.safetensors


FileNotFoundError: No such file or directory: "/tmp/tmpe3vf20_y/0-a6f5f2b1.safetensors"